# 03 — Landmark cohort and competing events

**Objective.** Regenerate the fixed early-stage risk set, horizon-aligned labels, overlap matrix, and financing/acquisition multi-state table.

**Scientific contract.** This notebook reports no empirical result until it executes successfully against hash-verified inputs. It writes immutable outputs plus a completion manifest. Expected counts are protocol assertions, not substituted observations.

Primary unit: one company. Absence of a recorded event is not labelled “failure.”

In [ ]:
# Standard CRUX-VC Colab bootstrap. Git dotfiles restore from the Drive project root; tokens never appear in cells.
import os, subprocess, sys
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    pass

import shutil
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/CRUX_Research")
for _dotfile in (".gitconfig", ".git-credentials"):
    if (DRIVE_PROJECT_ROOT / _dotfile).exists():
        shutil.copy(DRIVE_PROJECT_ROOT / _dotfile, Path.home() / _dotfile)
if (Path.home() / ".git-credentials").exists():
    os.chmod(Path.home() / ".git-credentials", 0o600)

REPO_URL = "https://github.com/anasbiswas1/crux-vc"
REPO_ROOT = Path(os.environ.get("CRUX_REPO_ROOT", "/content/drive/MyDrive/CRUX_Research/crux-vc"))
if not (REPO_ROOT / ".cruxvc-root").exists():
    if REPO_ROOT.exists() and any(REPO_ROOT.iterdir()):
        raise RuntimeError(f"{REPO_ROOT} exists but is not a CRUX-VC checkout")
    REPO_ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))

try:
    import yaml, pandas, sklearn, pyarrow  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO_ROOT / "requirements.txt")], check=True)

from cruxvc.runtime import bootstrap_notebook
CTX = bootstrap_notebook("03", suffix=None)
P, CFG, PROFILE = CTX.paths, CTX.config, CTX.profile

In [ ]:
from cruxvc.cohort import build_landmark_cohort
from cruxvc.io import read_json, read_table, write_json, write_table

funding_path = P.interim / "funding_events.parquet"
acquisitions_path = P.interim / "acquisitions_clean.parquet"
CTX.recorder.inputs.extend([funding_path, acquisitions_path])
funding = read_table(funding_path)
acquisitions = read_table(acquisitions_path)

In [ ]:
result = build_landmark_cohort(
    funding,
    acquisitions,
    landmark_start=CFG["cohort"]["landmark_start"],
    landmark_end=CFG["cohort"]["landmark_end"],
    administrative_cutoff=CFG["source"]["administrative_cutoff"],
)
cohort_path = write_table(result.cohort, P.processed / "cohort_labels.parquet")
flow_path = write_table(result.flow, P.audits / "03_cohort_flow.csv")
overlap_path = write_table(result.overlap, P.audits / "03_label_overlap.csv")
multistate_path = write_table(result.multistate, P.audits / "03_multistate_36.csv")
audit_path = write_json(result.audit, P.audits / "03_cohort_audit.json")

In [ ]:
source_manifest = read_json(P.protocol / "source_manifest.json")
if source_manifest["all_expected_hashes_match"] and CFG["execution"]["strict_expected_counts_when_hashes_match"]:
    expected_n = int(CFG["cohort"]["expected_n"])
    if len(result.cohort) != expected_n:
        raise RuntimeError(f"Cohort regeneration failed: expected {expected_n}, observed {len(result.cohort)}")
    expected_total = CFG["expected_block_counts"]["total"]
    for outcome in ["F18", "F36", "B+36", "C+36", "A36"]:
        observed = int(result.cohort[outcome].sum())
        if observed != int(expected_total[outcome]):
            raise RuntimeError(f"Total {outcome} count mismatch: expected {expected_total[outcome]}, observed {observed}")

In [ ]:
CTX.recorder.complete([cohort_path, flow_path, overlap_path, multistate_path, audit_path])
print(result.flow.to_string(index=False))
print(result.multistate.to_string(index=False))